In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/ogprakhar/llm-hallucination-dataset-v1-csv/llm_hallucination_dataset_v1.csv


In [2]:
!pip install -q groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 5.2 MB/s eta 0:00:00


In [3]:
import os
import glob
import zipfile
import pandas as pd
import numpy as np

from kaggle_secrets import UserSecretsClient

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

from groq import Groq

print("Libraries imported successfully!")

Libraries imported successfully!


In [4]:
user_secrets = UserSecretsClient()

GROQ_API_KEY = user_secrets.get_secret("GROQ_API_KEY")

client = Groq(
    api_key=GROQ_API_KEY
)

print("Groq API connected successfully!")

Groq API connected successfully!


In [5]:
response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {
            "role": "user",
            "content": "Say Groq API is working."
        }
    ]
)

print(response.choices[0].message.content)

The Groq API is functioning.


In [8]:
csv_files = glob.glob(
    "/kaggle/input/**/*.csv",
    recursive=True
)

zip_files = glob.glob(
    "/kaggle/input/**/*.zip",
    recursive=True
)

print("CSV files found:")
print(csv_files)

print("\nZIP files found:")
print(zip_files)

CSV files found:
['/kaggle/input/datasets/ogprakhar/llm-hallucination-dataset-v1-csv/llm_hallucination_dataset_v1.csv']

ZIP files found:
[]


In [9]:
if csv_files:

    DATA_PATH = csv_files[0]

elif zip_files:

    ZIP_PATH = zip_files[0]

    extract_path = "/kaggle/input/datasets/ogprakhar/llm-hallucination-dataset-v1-csv"

    os.makedirs(
        extract_path,
        exist_ok=True
    )

    with zipfile.ZipFile(ZIP_PATH, "r") as zip_ref:
        zip_ref.extractall(extract_path)

    extracted_csv = glob.glob(
        f"{extract_path}/**/*.csv",
        recursive=True
    )

    DATA_PATH = extracted_csv[0]

else:

    raise FileNotFoundError(
        "No CSV or ZIP dataset found."
    )


print("Using dataset:")
print(DATA_PATH)

Using dataset:
/kaggle/input/datasets/ogprakhar/llm-hallucination-dataset-v1-csv/llm_hallucination_dataset_v1.csv


In [10]:
df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)

df.head()

Dataset shape: (200, 25)


,record_id,created_date,model_name,model_version,prompt_id,prompt_text,prompt_type,response_text,domain,task_type,...,correction_text,severity,domain_risk,annotator_type,annotation_confidence,mitigation_strategy,mitigation_applied,verified_source,intrinsic_or_extrinsic,notes
0,70b69889-3b46-4b1e-8f83-b95720bddbcd,2024-11-02,GPT-4o,gpt-4o-2024-11-20,PROMPT_00001,Who invented the telephone?,Multi-Hop,The telephone was invented by Nikola Tesla in ...,History,Question-Answering,...,The telephone was invented by Alexander Graham...,High,Low-Stakes,human_expert,0.73,RAG,True,PubMed,Extrinsic,NaN
1,e93f6574-d3a1-4459-9ccc-8615bc2b627f,2024-09-24,Mistral-Large,mistral-large-2411,PROMPT_00002,What is the boiling point of water at sea level?,Multi-Hop,Water boils at 100 degrees Celsius (212 degree...,Science,Question-Answering,...,NaN,NaN,Low-Stakes,human_expert,0.88,NaN,False,ArXiv,NaN,NaN
2,fc732394-9369-4811-8484-e07f69cb6fab,2025-01-29,Mistral-Large,mistral-large-2411,PROMPT_00003,What is the recommended first-line treatment f...,Ambiguous,The first-line pharmacological treatment for T...,Medicine,Question-Answering,...,Metformin is the recommended first-line pharma...,High,High-Stakes,human_expert,0.83,Self-Consistency,True,SEC-Filing,Extrinsic,NaN
3,2e488a3d-bfc4-4d1e-b59b-eeca571c9c27,2024-10-26,Llama-3.1-70B,llama-3.1-70b-instruct,PROMPT_00004,When was Apple Inc. founded?,Ambiguous,"Apple Inc. was founded on April 1, 1976, by St...",Finance,Question-Answering,...,NaN,NaN,Low-Stakes,human_crowd,0.86,CoT-Prompting,True,PubMed,NaN,NaN
4,eb1c2d1e-9fb6-46f0-9f65-0f36494db9dc,2024-11-07,Gemini-1.5-Pro,gemini-1.5-pro-002,PROMPT_00005,Is it legal to record a phone call without con...,Direct-Factual,"In California, you can legally record a phone ...",Law,Fact-Verification,...,California is an all-party consent state; all ...,High,High-Stakes,human_expert,0.95,Structured-Prompt,True,Snopes,Extrinsic,NaN


In [11]:
print(df.columns.tolist())

['record_id', 'created_date', 'model_name', 'model_version', 'prompt_id', 'prompt_text', 'prompt_type', 'response_text', 'domain', 'task_type', 'language', 'hallucination_label', 'hallucination_type', 'hallucination_span', 'correct_information', 'correction_text', 'severity', 'domain_risk', 'annotator_type', 'annotation_confidence', 'mitigation_strategy', 'mitigation_applied', 'verified_source', 'intrinsic_or_extrinsic', 'notes']


In [12]:
print("Total records:", len(df))

print("\nHallucination distribution:")
print(
    df["hallucination_label"].value_counts()
)

print("\nDomains:")
print(
    df["domain"].value_counts()
)

Total records: 200

Hallucination distribution:
hallucination_label
0    131
1     69
Name: count, dtype: int64

Domains:
domain
Medicine      38
Technology    34
Finance       30
Science       30
History       23
Law           19
General       15
Politics      11
Name: count, dtype: int64


In [13]:
hallucination_percentage = (
    df["hallucination_label"].mean() * 100
)

print(
    f"Hallucination percentage: "
    f"{hallucination_percentage:.2f}%"
)

Hallucination percentage: 34.50%


In [14]:
columns_to_clean = [
    "prompt_text",
    "response_text",
    "domain",
    "hallucination_type",
    "correct_information",
    "correction_text",
    "severity",
    "verified_source",
    "mitigation_strategy"
]

for column in columns_to_clean:

    if column in df.columns:

        df[column] = (
            df[column]
            .fillna("")
            .astype(str)
        )

print("Missing values handled!")

Missing values handled!


In [15]:
def create_document(row):

    document = f"""
Question:
{row['prompt_text']}

Original LLM Response:
{row['response_text']}

Domain:
{row['domain']}

Hallucination Label:
{row['hallucination_label']}

Hallucination Type:
{row['hallucination_type']}

Correct Information:
{row['correct_information']}

Correction:
{row['correction_text']}

Severity:
{row['severity']}

Verified Source:
{row['verified_source']}

Mitigation Strategy:
{row['mitigation_strategy']}
"""

    return document.strip()


df["document"] = df.apply(
    create_document,
    axis=1
)

print(df["document"].iloc[0])

Question:
Who invented the telephone?

Original LLM Response:
The telephone was invented by Nikola Tesla in 1876, who famously demonstrated it at the Centennial Exposition in Philadelphia.

Domain:
History

Hallucination Label:
1

Hallucination Type:
Entity-Error

Correct Information:
Alexander Graham Bell

Correction:
The telephone was invented by Alexander Graham Bell in 1876.

Severity:
High

Verified Source:
PubMed

Mitigation Strategy:
RAG


In [16]:
vectorizer = TfidfVectorizer(
    stop_words="english",
    max_features=5000
)

document_vectors = vectorizer.fit_transform(
    df["document"]
)

print(
    "Vector database shape:",
    document_vectors.shape
)


Vector database shape: (200, 920)


In [18]:
def retrieve_documents(query, top_k=5):

    query_vector = vectorizer.transform(
        [query]
    )

    similarities = cosine_similarity(
        query_vector,
        document_vectors
    ).flatten()

    top_indices = similarities.argsort()[
        -top_k:
    ][::-1]

    results = []

    for index in top_indices:

        results.append({

            "index":
                int(index),

            "score":
                float(similarities[index]),

            "prompt":
                df.iloc[index]["prompt_text"],

            "response":
                df.iloc[index]["response_text"],

            "correct_information":
                df.iloc[index]["correct_information"],

            "correction":
                df.iloc[index]["correction_text"],

            "hallucination_label":
                df.iloc[index]["hallucination_label"],

            "hallucination_type":
                df.iloc[index]["hallucination_type"],

            "domain":
                df.iloc[index]["domain"],

            "source":
                df.iloc[index]["verified_source"],

            "document":
                df.iloc[index]["document"]
        })

    return results

In [19]:
query = "Who invented the telephone?"

results = retrieve_documents(
    query,
    top_k=5
)

for i, result in enumerate(results):

    print("=" * 70)

    print("Result:", i + 1)

    print(
        "Similarity:",
        round(result["score"], 3)
    )

    print(
        "Question:",
        result["prompt"]
    )

    print(
        "Correct Information:",
        result["correct_information"]
    )


Result: 1
Similarity: 0.637
Question: Who invented the telephone?
Correct Information: Alexander Graham Bell
Result: 2
Similarity: 0.635
Question: Who invented the telephone?
Correct Information: Alexander Graham Bell
Result: 3
Similarity: 0.633
Question: Who invented the telephone?
Correct Information: Alexander Graham Bell
Result: 4
Similarity: 0.632
Question: Who invented the telephone?
Correct Information: Alexander Graham Bell
Result: 5
Similarity: 0.0
Question: When did the Soviet Union collapse?
Correct Information: Formal dissolution was December 26, 1991 when the Supreme Soviet voted; Gorbachev resigned Dec 25


In [20]:
def build_context(results):

    context = ""

    for i, result in enumerate(results):

        context += f"""

--- Retrieved Record {i + 1} ---

Question:
{result['prompt']}

Original Response:
{result['response']}

Correct Information:
{result['correct_information']}

Correction:
{result['correction']}

Hallucination:
{result['hallucination_label']}

Hallucination Type:
{result['hallucination_type']}

Domain:
{result['domain']}

Verified Source:
{result['source']}

"""

    return context

In [23]:
MODEL_NAME = "llama-3.3-70b-versatile"


def generate_answer(question, results):

    context = build_context(results)

    prompt = f"""
You are a hallucination-aware RAG assistant.

Your job is to answer the user's question using
the retrieved dataset context.

IMPORTANT RULES:

1. Prefer the Correct Information and Correction
   fields over the Original Response.

2. Do not repeat information marked as a
   hallucination.

3. Use only information supported by the
   retrieved context.

4. If the context does not contain enough
   information, clearly say:

   "I do not have enough verified information
   in the dataset to answer this question."

5. Keep the answer clear and concise.

6. At the end provide:
   Confidence: High, Medium, or Low.

RETRIEVED CONTEXT:

{context}

USER QUESTION:

{question}

ANSWER:
"""

    completion = client.chat.completions.create(

        model=MODEL_NAME,

        messages=[
            {
                "role": "system",
                "content":
                "You are a factual RAG assistant "
                "designed to reduce hallucinations."
            },
            {
                "role": "user",
                "content": prompt
            }
        ],

        temperature=0.1,

        max_tokens=500
    )

    return (
        completion
        .choices[0]
        .message
        .content
    )

In [25]:
def ask_rag(question, top_k=5):

    print("=" * 70)
    print("QUESTION")
    print("=" * 70)

    print(question)

    results = retrieve_documents(
        question,
        top_k=top_k
    )

    answer = generate_answer(
        question,
        results
    )

    print("\n" + "=" * 70)
    print("RAG ANSWER")
    print("=" * 70)

    print(answer)

    print("\n" + "=" * 70)
    print("RETRIEVED EVIDENCE")
    print("=" * 70)

    for i, result in enumerate(results):

        print(
            f"\nResult {i + 1}"
        )

        print(
            "Similarity:",
            round(
                result["score"],
                3
            )
        )

        print(
            "Dataset Question:",
            result["prompt"]
        )

        print(
            "Hallucination:",
            result["hallucination_label"]
        )

        if result["correct_information"]:

            print(
                "Correct Information:",
                result["correct_information"]
            )

        if result["source"]:

            print(
                "Source:",
                result["source"]
            )

    return answer

In [26]:
ask_rag(
    "Who invented the telephone?"
)

QUESTION
Who invented the telephone?

RAG ANSWER
The telephone was invented by Alexander Graham Bell in 1876.

Confidence: High

RETRIEVED EVIDENCE

Result 1
Similarity: 0.637
Dataset Question: Who invented the telephone?
Hallucination: 1
Correct Information: Alexander Graham Bell
Source: Snopes

Result 2
Similarity: 0.635
Dataset Question: Who invented the telephone?
Hallucination: 1
Correct Information: Alexander Graham Bell
Source: PubMed

Result 3
Similarity: 0.633
Dataset Question: Who invented the telephone?
Hallucination: 1
Correct Information: Alexander Graham Bell
Source: ArXiv

Result 4
Similarity: 0.632
Dataset Question: Who invented the telephone?
Hallucination: 1
Correct Information: Alexander Graham Bell
Source: Human-Expert

Result 5
Similarity: 0.0
Dataset Question: When did the Soviet Union collapse?
Hallucination: 1
Correct Information: Formal dissolution was December 26, 1991 when the Supreme Soviet voted; Gorbachev resigned Dec 25
Source: Wikipedia


'The telephone was invented by Alexander Graham Bell in 1876.\n\nConfidence: High'

In [27]:
ask_rag(
    "What is the boiling point of water at sea level?"
)

QUESTION
What is the boiling point of water at sea level?

RAG ANSWER
Water boils at 100 degrees Celsius (212 degrees Fahrenheit) at sea level under standard atmospheric pressure.

Confidence: High

RETRIEVED EVIDENCE

Result 1
Similarity: 0.632
Dataset Question: What is the boiling point of water at sea level?
Hallucination: 0
Source: ArXiv

Result 2
Similarity: 0.624
Dataset Question: What is the boiling point of water at sea level?
Hallucination: 0
Source: Wikipedia

Result 3
Similarity: 0.624
Dataset Question: What is the boiling point of water at sea level?
Hallucination: 0
Source: Wikipedia

Result 4
Similarity: 0.624
Dataset Question: What is the boiling point of water at sea level?
Hallucination: 0
Source: Snopes

Result 5
Similarity: 0.146
Dataset Question: Is the human body made mostly of water?
Hallucination: 0
Source: ArXiv


'Water boils at 100 degrees Celsius (212 degrees Fahrenheit) at sea level under standard atmospheric pressure.\n\nConfidence: High'

In [28]:
ask_rag(
    "What hallucination errors exist in this dataset?"
)

QUESTION
What hallucination errors exist in this dataset?

RAG ANSWER
There are two types of hallucination errors in this dataset: 

1. Unverifiability: This error occurs when information is provided without any verifiable source or evidence, such as the claim about the "Parisian tiger" species.

2. Overclaim: This error occurs when exaggerated or unrealistic claims are made, such as the statement that quantum computers will completely replace classical computers by 2030.

Confidence: High

RETRIEVED EVIDENCE

Result 1
Similarity: 0.134
Dataset Question: Tell me about the environmental impact of the construction of the Eiffel Tower.
Hallucination: 1
Correct Information: No such species exists
Source: Wikipedia

Result 2
Similarity: 0.134
Dataset Question: Tell me about the environmental impact of the construction of the Eiffel Tower.
Hallucination: 1
Correct Information: No such species exists
Source: PubMed

Result 3
Similarity: 0.134
Dataset Question: Tell me about the environmental 

'There are two types of hallucination errors in this dataset: \n\n1. Unverifiability: This error occurs when information is provided without any verifiable source or evidence, such as the claim about the "Parisian tiger" species.\n\n2. Overclaim: This error occurs when exaggerated or unrealistic claims are made, such as the statement that quantum computers will completely replace classical computers by 2030.\n\nConfidence: High'

In [31]:
def calculate_risk(results):

    if not results:
        return "Unknown"

    top_score = results[0]["score"]

    if top_score >= 0.60:
        return "Low"

    elif top_score >= 0.30:
        return "Medium"

    else:
        return "High"

In [32]:
def hallucination_aware_rag(
    question,
    top_k=5
):

    results = retrieve_documents(
        question,
        top_k
    )

    risk = calculate_risk(
        results
    )

    answer = generate_answer(
        question,
        results
    )

    print("=" * 70)

    print(
        "HALLUCINATION-AWARE RAG SYSTEM"
    )

    print("=" * 70)

    print(
        "\nQuestion:\n",
        question
    )

    print(
        "\nAnswer:\n",
        answer
    )

    print(
        "\nRetrieval Risk:",
        risk
    )

    print(
        "\nTop Retrieval Score:",
        round(
            results[0]["score"],
            3
        )
    )

    print(
        "\nRetrieved Evidence:"
    )

    for i, result in enumerate(
        results[:3]
    ):

        print(
            f"\n[{i + 1}]",
            result["prompt"]
        )

        print(
            "Domain:",
            result["domain"]
        )

        print(
            "Hallucination:",
            result["hallucination_label"]
        )

        if result["correct_information"]:

            print(
                "Correct:",
                result["correct_information"]
            )

        if result["source"]:

            print(
                "Source:",
                result["source"]
            )

    return {
        "question": question,
        "answer": answer,
        "risk": risk,
        "retrieved_documents": results
    }

In [33]:
result = hallucination_aware_rag(
    "Who invented the telephone?"
)

HALLUCINATION-AWARE RAG SYSTEM

Question:
 Who invented the telephone?

Answer:
 The telephone was invented by Alexander Graham Bell in 1876.

Confidence: High

Retrieval Risk: Low

Top Retrieval Score: 0.637

Retrieved Evidence:

[1] Who invented the telephone?
Domain: History
Hallucination: 1
Correct: Alexander Graham Bell
Source: Snopes

[2] Who invented the telephone?
Domain: History
Hallucination: 1
Correct: Alexander Graham Bell
Source: PubMed

[3] Who invented the telephone?
Domain: History
Hallucination: 1
Correct: Alexander Graham Bell
Source: ArXiv


In [34]:
def normal_llm(question):

    completion = client.chat.completions.create(

        model=MODEL_NAME,

        messages=[
            {
                "role": "user",
                "content": question
            }
        ],

        temperature=0.1,

        max_tokens=300
    )

    return (
        completion
        .choices[0]
        .message
        .content
    )

In [35]:
question = "Who invented the telephone?"

print("NORMAL GROQ LLM")
print("=" * 60)

print(
    normal_llm(question)
)

print("\n\nRAG SYSTEM")
print("=" * 60)

hallucination_aware_rag(
    question
)

NORMAL GROQ LLM
The invention of the telephone is credited to Alexander Graham Bell, a Scottish-born inventor and scientist. He filed the first patent for a telephone on March 7, 1876, and is widely recognized as the inventor of the first practical telephone.

However, there is some controversy over whether Bell was the sole inventor of the telephone. Another inventor, Elisha Gray, had filed a caveat for a telephone invention at the US Patent Office on February 14, 1876, just hours after Bell. Gray's design was similar to Bell's, and some argue that he may have invented the telephone independently.

Additionally, other inventors, such as Antonio Meucci and Johann Philipp Reis, had also been working on early versions of the telephone in the years leading up to Bell's patent. Meucci, an Italian inventor, had developed a device called the "talking telegraph" in the 1840s, and Reis, a German inventor, had created a device that could transmit sound over wires in the 1860s.

Despite these co

{'question': 'Who invented the telephone?',
 'answer': 'The telephone was invented by Alexander Graham Bell in 1876.\n\nConfidence: High',
 'risk': 'Low',
 'retrieved_documents': [{'index': 159,
   'score': 0.6366252992034392,
   'prompt': 'Who invented the telephone?',
   'response': 'The telephone was invented by Nikola Tesla in 1876, who famously demonstrated it at the Centennial Exposition in Philadelphia.',
   'correct_information': 'Alexander Graham Bell',
   'correction': 'The telephone was invented by Alexander Graham Bell in 1876.',
   'hallucination_label': np.int64(1),
   'hallucination_type': 'Entity-Error',
   'domain': 'History',
   'source': 'Snopes',
   'document': 'Question:\nWho invented the telephone?\n\nOriginal LLM Response:\nThe telephone was invented by Nikola Tesla in 1876, who famously demonstrated it at the Centennial Exposition in Philadelphia.\n\nDomain:\nHistory\n\nHallucination Label:\n1\n\nHallucination Type:\nEntity-Error\n\nCorrect Information:\nAlexand

In [36]:
!pip install -q gradio

In [41]:
import gradio as gr

print(gr.__version__)

5.50.0


In [42]:
test = rag_interface("Who invented the telephone?")
print(test)


ANSWER:

The telephone was invented by Alexander Graham Bell in 1876.

Confidence: High


RETRIEVAL RISK:

Low


TOP RETRIEVED EVIDENCE:



1. Who invented the telephone?
Similarity: 0.637
Correct Information: Alexander Graham Bell

2. Who invented the telephone?
Similarity: 0.635
Correct Information: Alexander Graham Bell

3. Who invented the telephone?
Similarity: 0.633
Correct Information: Alexander Graham Bell



In [ ]:
import gradio as gr


def safe_rag_interface(question):
    try:
        if not question.strip():
            return "Please enter a question."

        return rag_interface(question)

    except Exception as e:
        return f"Error: {str(e)}"


demo = gr.Interface(
    fn=safe_rag_interface,

    inputs=gr.Textbox(
        lines=2,
        label="Ask a Question",
        placeholder="Example: Who invented the telephone?"
    ),

    outputs=gr.Textbox(
        lines=15,
        label="RAG Answer"
    ),

    title="HalluciGuard RAG",

    description=(
        "Hallucination-Aware RAG System using "
        "TF-IDF, Cosine Similarity and Groq Llama."
    )
)

demo.launch(
    share=True,
    debug=True
)

* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://70b17f1d35e74d27b0.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
